In [1]:
%load_ext autoreload
%autoreload 2

import requests
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm.auto import tqdm
import time
import matplotlib.pyplot as plt

from utils import (
    fetch_market_data, parse_future_name, bin_orderbook,
    compute_segment_carry, interpolate_forward, compute_continuous_forward_curve
)

In [29]:
start_date = datetime(2025, 8, 1)
end_date = datetime(2025, 8, 2)
# Fetch SPOT data for the same period with depth=0 (only mid prices)
spot_df = fetch_market_data('6', 'SPOT', 'USDT-USDC', start_date, end_date, 'daily', depth=2)

spot_df['mid_price'] = (spot_df['bid_1_px'] + spot_df['ask_1_px']) / 2

Fetching SPOT data (module=6) for USDT-USDC
Period: 2025-08-01 00:00:00 to 2025-08-02 00:00:00
Split into 1 requests


Fetching data:   0%|          | 0/1 [00:00<?, ?it/s]

IndexError: list index out of range

In [28]:
print(spot_df['symbol'].unique())
def parse_spot_symbol(symbol):
    return symbol.split('.')[0].split('-')




['USDG-USDT.OK' 'USDT-TRY.OK' 'USDT-AED.OK' 'USDC-BRL.OK' 'USDT-BRL.OK'
 'USDT-USD.OK' 'USDC-AUD.OK' 'USDT-AUD.OK' 'USDC-USDT.OK' 'USDT-SGD.OK'
 'USDC-SGD.OK' 'USDT-EUR.OK' 'USDC-EUR.OK']


In [9]:
# Calculate memory usage in MB
memory_mb = spot_df.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Memory usage of spot_df: {memory_mb:.2f} MB")

Memory usage of spot_df: 4948.53 MB


In [13]:
for col in spot_df.columns:
    print(f"{col}: {spot_df[col].dtype}, {spot_df[col].nunique()} unique values, with {spot_df[col].isna().sum()} NA")

print(spot_df.head())

timeMs: int64, 27875362 unique values, with 0 NA
exchTimeMs: int64, 27868058 unique values, with 0 NA
bid_1_px: float64, 59023 unique values, with 0 NA
bid_1_qty: float64, 2279677 unique values, with 0 NA
bid_1_ordCnt: int64, 251 unique values, with 0 NA
ask_1_px: float64, 59363 unique values, with 0 NA
ask_1_qty: float64, 2203702 unique values, with 0 NA
ask_1_ordCnt: int64, 281 unique values, with 0 NA
bid_2_px: float64, 60816 unique values, with 0 NA
bid_2_qty: float64, 754520 unique values, with 0 NA
bid_2_ordCnt: int64, 92 unique values, with 0 NA
ask_2_px: float64, 60887 unique values, with 0 NA
ask_2_qty: float64, 730844 unique values, with 0 NA
ask_2_ordCnt: int64, 106 unique values, with 0 NA
symbol: object, 3 unique values, with 0 NA
mid_price: float64, 151330 unique values, with 0 NA
          timeMs     exchTimeMs  bid_1_px  bid_1_qty  bid_1_ordCnt  ask_1_px  \
0  1735689600018  1735689600009   93467.8   0.048144             2   93480.6   
1  1735689600031  1735689600019   

# Does OKX have a minimum tick size on SPOT markets?

In [12]:
# Calculate minimum tick size by looking at price differences between adjacent levels
# Do this separately for each symbol
for symbol in spot_df['symbol'].unique():
    symbol_df = spot_df[spot_df['symbol'] == symbol]
    
    # Calculate differences between adjacent levels
    bid_diffs = symbol_df['bid_1_px'] - symbol_df['bid_2_px'] 
    ask_diffs = symbol_df['ask_2_px'] - symbol_df['ask_1_px']
    
    # Filter out zeros and NaNs to find true minimum tick
    min_bid_diff = bid_diffs[bid_diffs > 0].min()
    min_ask_diff = ask_diffs[ask_diffs > 0].min()
    
    min_tick = min(min_bid_diff, min_ask_diff)
    print(f"Minimum tick size for {symbol}: {min_tick}")



Minimum tick size for BTC-USD.OK: 0.09999999999126885
Minimum tick size for BTC-USDC.OK: 0.09999999999126885
Minimum tick size for BTC-USDT.OK: 0.09999999999126885


# Verify no arbitrage
We will check if there is static arbitrage in the different numeraries present for SPOT data.

We will first use BID-ASK TAKER-TAKER canonical trades, accounting for fees:

In [11]:
fee_structure = {
    'Regular': {
        'maker_fee': 0.00200,  # 0.200%
        'taker_fee': 0.00350   # 0.350%
    },
    'VIP 1': {
        'maker_fee': 0.00180,  # 0.180%
        'taker_fee': 0.00300   # 0.300%
    },
    'VIP 2': {
        'maker_fee': 0.00150,  # 0.150%
        'taker_fee': 0.00280   # 0.280%
    },
    'VIP 3': {
        'maker_fee': 0.00100,  # 0.100%
        'taker_fee': 0.00250   # 0.250%
    },
    'VIP 4': {
        'maker_fee': 0.00050,  # 0.050%
        'taker_fee': 0.00230   # 0.230%
    },
    'VIP 5': {
        'maker_fee': 0.00000,  # 0.000%
        'taker_fee': 0.00200   # 0.200%
    },
    'VIP 6': {
        'maker_fee': -0.00030, # -0.030%
        'taker_fee': 0.00150   # 0.150%
    },
    'VIP 7': {
        'maker_fee': -0.00050, # -0.050%
        'taker_fee': 0.00100   # 0.100%
    },
    'VIP 8': {
        'maker_fee': -0.00050, # -0.050%
        'taker_fee': 0.00080   # 0.080%
    }
}

# Print fees for each VIP level
for level, fees in fee_structure.items():
    print(f"{level}: maker={fees['maker_fee']*100:.3f}%, taker={fees['taker_fee']*100:.3f}%")


Regular: maker=0.200%, taker=0.350%
VIP 1: maker=0.180%, taker=0.300%
VIP 2: maker=0.150%, taker=0.280%
VIP 3: maker=0.100%, taker=0.250%
VIP 4: maker=0.050%, taker=0.230%
VIP 5: maker=0.000%, taker=0.200%
VIP 6: maker=-0.030%, taker=0.150%
VIP 7: maker=-0.050%, taker=0.100%
VIP 8: maker=-0.050%, taker=0.080%


In [17]:
def find_arbitrage_opportunities(spot_df, maker_fee, taker_fee):
    """
    Calculate arbitrage opportunities between symbol pairs accounting for trading fees
    
    Args:
        spot_df: DataFrame with spot market data
        maker_fee: Maker fee as decimal (e.g. 0.001 for 0.1%)
        taker_fee: Taker fee as decimal (e.g. 0.001 for 0.1%)
        
    Returns:
        List of dicts containing arbitrage statistics
    """
    symbols = spot_df['symbol'].unique()
    symbol_pairs = [(s1, s2) for i, s1 in enumerate(symbols) for s2 in symbols[i+1:]]
    results = []
    
    for s1, s2 in symbol_pairs:
        # Get prices and quantities for both symbols
        df1 = spot_df[spot_df['symbol'] == s1][['timeMs', 'bid_1_px', 'ask_1_px', 'bid_1_qty', 'ask_1_qty']]
        df2 = spot_df[spot_df['symbol'] == s2][['timeMs', 'bid_1_px', 'ask_1_px', 'bid_1_qty', 'ask_1_qty']]
        
        # Merge on timeMs to compare contemporaneous snapshots
        merged = pd.merge(df1, df2, on='timeMs', suffixes=('_1', '_2'))
        
        # Account for taker fees in both directions
        effective_bid_1 = merged['bid_1_px_1'] * (1 - taker_fee)  # Selling incurs taker fee
        effective_ask_2 = merged['ask_1_px_2'] * (1 + taker_fee)  # Buying incurs taker fee
        effective_bid_2 = merged['bid_1_px_2'] * (1 - taker_fee)
        effective_ask_1 = merged['ask_1_px_1'] * (1 + taker_fee)
        
        # Check for arbitrage opportunities after fees
        arb_1_to_2 = effective_bid_1 > effective_ask_2  # Can buy on s2 and sell on s1
        arb_2_to_1 = effective_bid_2 > effective_ask_1  # Can buy on s1 and sell on s2
        
        # Calculate arbitrage profits as percentages after fees
        arb_1_to_2_profit_pct = 100 * (effective_bid_1[arb_1_to_2] - effective_ask_2[arb_1_to_2]) / effective_ask_2[arb_1_to_2]
        arb_2_to_1_profit_pct = 100 * (effective_bid_2[arb_2_to_1] - effective_ask_1[arb_2_to_1]) / effective_ask_1[arb_2_to_1]
        
        # Calculate total profit by considering minimum quantities
        arb_1_to_2_qty = np.minimum(merged['bid_1_qty_1'][arb_1_to_2], merged['ask_1_qty_2'][arb_1_to_2])
        arb_2_to_1_qty = np.minimum(merged['bid_1_qty_2'][arb_2_to_1], merged['ask_1_qty_1'][arb_2_to_1])
        
        arb_1_to_2_profit = arb_1_to_2_qty * (effective_bid_1[arb_1_to_2] - effective_ask_2[arb_1_to_2])
        arb_2_to_1_profit = arb_2_to_1_qty * (effective_bid_2[arb_2_to_1] - effective_ask_1[arb_2_to_1])
        
        results.append({
            'pair': f"{s1} vs {s2}",
            'arb_1_to_2_count': arb_1_to_2.sum(),
            'arb_1_to_2_max_profit': arb_1_to_2_profit_pct.max() if len(arb_1_to_2_profit_pct) > 0 else 0,
            'arb_1_to_2_total_profit': arb_1_to_2_profit.sum() if len(arb_1_to_2_profit) > 0 else 0,
            'arb_2_to_1_count': arb_2_to_1.sum(),
            'arb_2_to_1_max_profit': arb_2_to_1_profit_pct.max() if len(arb_2_to_1_profit_pct) > 0 else 0,
            'arb_2_to_1_total_profit': arb_2_to_1_profit.sum() if len(arb_2_to_1_profit) > 0 else 0
        })
    
    return results

# Check arbitrage opportunities for top 3 VIP levels
top_3_levels = list(fee_structure.items())[-3:]
for level, fees in top_3_levels:
    print(f"\nAnalyzing arbitrage opportunities for {level}:")
    print(f"Maker fee: {fees['maker_fee']*100:.3f}%, Taker fee: {fees['taker_fee']*100:.3f}%")
    
    results = find_arbitrage_opportunities(spot_df, fees['maker_fee'], fees['taker_fee'])
    
    total_opportunities = 0
    total_profit = 0
    max_profit = 0
    
    for r in results:
        print(f"\n{r['pair']}:")
        print(f"  {r['pair'].split(' vs ')[0]}->{r['pair'].split(' vs ')[1]} arbitrage opportunities: {r['arb_1_to_2_count']}")
        if r['arb_1_to_2_max_profit'] > 0:
            print(f"    Max profit: {r['arb_1_to_2_max_profit']:.2f}%")
            print(f"    Total profit: {r['arb_1_to_2_total_profit']:.2f}")
        print(f"  {r['pair'].split(' vs ')[1]}->{r['pair'].split(' vs ')[0]} arbitrage opportunities: {r['arb_2_to_1_count']}")
        if r['arb_2_to_1_max_profit'] > 0:
            print(f"    Max profit: {r['arb_2_to_1_max_profit']:.2f}%")
            print(f"    Total profit: {r['arb_2_to_1_total_profit']:.2f}")
            
        # Update summary statistics
        total_opportunities += r['arb_1_to_2_count'] + r['arb_2_to_1_count']
        total_profit += r['arb_1_to_2_total_profit'] + r['arb_2_to_1_total_profit']
        max_profit = max(max_profit, r['arb_1_to_2_max_profit'], r['arb_2_to_1_max_profit'])
    
    print(f"\nSummary for {level}:")
    print(f"  Total arbitrage opportunities: {total_opportunities}")
    print(f"  Total profit across all pairs: {total_profit:.2f}")
    print(f"  Maximum profit percentage: {max_profit:.2f}%")


Analyzing arbitrage opportunities for VIP 6:
Maker fee: -0.030%, Taker fee: 0.150%

BTC-USD.OK vs BTC-USDC.OK:
  BTC-USD.OK->BTC-USDC.OK arbitrage opportunities: 0
  BTC-USDC.OK->BTC-USD.OK arbitrage opportunities: 0

BTC-USD.OK vs BTC-USDT.OK:
  BTC-USD.OK->BTC-USDT.OK arbitrage opportunities: 0
  BTC-USDT.OK->BTC-USD.OK arbitrage opportunities: 0

BTC-USDC.OK vs BTC-USDT.OK:
  BTC-USDC.OK->BTC-USDT.OK arbitrage opportunities: 0
  BTC-USDT.OK->BTC-USDC.OK arbitrage opportunities: 0

Summary for VIP 6:
  Total arbitrage opportunities: 0
  Total profit across all pairs: 0.00
  Maximum profit percentage: 0.00%

Analyzing arbitrage opportunities for VIP 7:
Maker fee: -0.050%, Taker fee: 0.100%

BTC-USD.OK vs BTC-USDC.OK:
  BTC-USD.OK->BTC-USDC.OK arbitrage opportunities: 0
  BTC-USDC.OK->BTC-USD.OK arbitrage opportunities: 0

BTC-USD.OK vs BTC-USDT.OK:
  BTC-USD.OK->BTC-USDT.OK arbitrage opportunities: 0
  BTC-USDT.OK->BTC-USD.OK arbitrage opportunities: 0

BTC-USDC.OK vs BTC-USDT.OK:
  

In [ ]:
spot_df_2 = fetch_market_data('6', 'SPOT', 'BTC-USD', start_date, end_date, 'daily', depth=2)

Fetching SPOT data (module=6) for BTC-USD
Period: 2025-01-01 00:00:00 to 2025-01-04 00:00:00
Split into 3 requests


Fetching data:   0%|          | 0/3 [00:00<?, ?it/s]

Fetch #1/3: 3 files found | Total: 3 files, 456.71 MB
